In [30]:
# ============================================================
# Smoothness fine-tune candidates: complete TestData evaluation
#
# 評価対象:
#   SMOOTH_WEIGHT = 0.001 / 0.01 / 0.1
#
# 出力:
#   - 各候補・各テストペアの Dice / Jaccard / MS-SSIM /
#     FSIM / RMSE / NCC / Folding / IC MSE
#   - 各候補の mean ± SD
#   - 平均Folding、最大Folding、最悪ペア
#   - 3候補の比較CSV
# ============================================================

import os
os.environ["VXM_BACKEND"] = "pytorch"

import csv
import gc
import math
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import voxelmorph as vxm

from piq import fsim, multi_scale_ssim
from tabulate import tabulate


# ------------------------------------------------------------
# 1. Paths / evaluation settings
# ------------------------------------------------------------

TEST_DATA_ROOT = Path(
    r"D:\Saito\Data\TestData"
)

CANDIDATE_ROOT = Path(
    r"D:\Saito\fullres_trilinear_ic_beta1e3_80k"
    r"\finetune_smoothness_comparison"
)

CANDIDATE_MODELS = {
    "smooth_1e-03": (
        CANDIDATE_ROOT
        / "smooth_1e-03"
        / (
            "model_fullres_trilinear_ic_"
            "smooth_1e-03_finetuned_5000k_final.pth"
        )
    ),
    "smooth_2e-03": (
        CANDIDATE_ROOT
        / "smooth_2e-03"
        / (
            "model_fullres_trilinear_ic_"
            "smooth_2e-03_finetuned_5000k_final.pth"
        )
    ),
    "smooth_3e-03": (
        CANDIDATE_ROOT
        / "smooth_3e-03"
        / (
            "model_fullres_trilinear_ic_"
            "smooth_3e-03_finetuned_5000k_final.pth"
        )
    ),
    "smooth_5e-03": (
        CANDIDATE_ROOT
        / "smooth_5e-03"
        / (
            "model_fullres_trilinear_ic_"
            "smooth_5e-03_finetuned_5000k_final.pth"
        )
    ),
    "smooth_7e-03": (
        CANDIDATE_ROOT
        / "smooth_7e-03"
        / (
            "model_fullres_trilinear_ic_"
            "smooth_7e-03_finetuned_5000k_final.pth"
        )
    ),
    "smooth_8e-03": (
        CANDIDATE_ROOT
        / "smooth_8e-03"
        / (
            "model_fullres_trilinear_ic_"
            "smooth_8e-03_finetuned_5000k_final.pth"
        )
    ),
    "smooth_1e-02": (
        CANDIDATE_ROOT
        / "smooth_1e-02"
        / (
            "model_fullres_trilinear_ic_"
            "smooth_1e-02_finetuned_5000k_final.pth"
        )
    ),
    "smooth_1e-01": (
        CANDIDATE_ROOT
        / "smooth_1e-01"
        / (
            "model_fullres_trilinear_ic_"
            "smooth_1e-01_finetuned_5000k_final.pth"
        )
    ),
}


RESULT_ROOT = (
    CANDIDATE_ROOT
    / "testdata_comparison"
)

RESULT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# TestData/<pair>/ 内のファイル名ソート順
FIXED_FILE_INDEX = 0
MOVING_FILE_INDEX = 1

# Noneなら全128スライスを評価
EVAL_SLICE_START = None
EVAL_SLICE_END = None

# Dice / Jaccardの前景閾値
FOREGROUND_THRESHOLD = 0.144

# 正規化値をHUへ変換する範囲
CT_MIN = -1200.0
CT_MAX = 3146.98

FULL_SIZE = (128, 256, 256)
LOW_SIZE = (64, 128, 128)

NB_FEATURES = [
    [32, 64, 64, 64, 64],
    [64, 64, 64, 64, 64, 32, 16, 16],
]

device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)
print("TestData:", TEST_DATA_ROOT)
print("Result directory:", RESULT_ROOT)

if not TEST_DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f"TestDataが見つかりません: {TEST_DATA_ROOT}"
    )

for name, path in CANDIDATE_MODELS.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"{name} の重みが見つかりません:\n{path}"
        )


# ------------------------------------------------------------
# 2. Same Haar / model definitions as training
# ------------------------------------------------------------

class Haar3DAnalysisFast(nn.Module):
    def __init__(self):
        super().__init__()

        h_low = torch.tensor(
            [1.0, 1.0],
            dtype=torch.float32,
        ) / math.sqrt(2.0)

        h_high = torch.tensor(
            [1.0, -1.0],
            dtype=torch.float32,
        ) / math.sqrt(2.0)

        filters = []

        for z_filter in [h_low, h_high]:
            for y_filter in [h_low, h_high]:
                for x_filter in [h_low, h_high]:
                    kernel = (
                        z_filter[:, None, None]
                        * y_filter[None, :, None]
                        * x_filter[None, None, :]
                    )

                    filters.append(kernel)

        weight = torch.stack(
            filters,
            dim=0,
        ).unsqueeze(1)

        self.register_buffer(
            "weight",
            weight,
        )

    def forward(self, image):
        image = F.pad(
            image,
            (0, 1, 0, 1, 0, 1),
        )

        return F.conv3d(
            image,
            self.weight,
            stride=2,
            padding=0,
        )


analysis_fast = Haar3DAnalysisFast().to(
    device
).eval()

transformer_full = vxm.layers.SpatialTransformer(
    FULL_SIZE
).to(device)

transformer_low = vxm.layers.SpatialTransformer(
    LOW_SIZE
).to(device)


def wavelet_input(image):
    return analysis_fast(image)


def low_dvf_to_full_trilinear(flow_low):
    return F.interpolate(
        flow_low.float(),
        size=FULL_SIZE,
        mode="trilinear",
        align_corners=False,
    ) * 2.0


def create_model():
    return vxm.networks.VxmDense_128_256_256(
        FULL_SIZE,
        NB_FEATURES,
        int_steps=0,
    ).to(device)


# ------------------------------------------------------------
# 3. TestData loading
# ------------------------------------------------------------

def to_dhw(array, source_name):
    """
    (H, W, D, N) または (D, H, W) を
    (D, H, W) = (128, 256, 256) に統一する。
    """
    array = np.asarray(
        array,
        dtype=np.float32,
    )

    if array.ndim == 3:
        candidates = [
            array,
            np.transpose(array, (2, 0, 1)),
        ]

    elif array.ndim == 4:
        candidates = [
            np.transpose(
                array[..., 0],
                (2, 0, 1),
            ),
            array[0],
        ]

    else:
        raise ValueError(
            f"{source_name}: shapeが不正です: "
            f"{array.shape}"
        )

    for candidate in candidates:
        if tuple(candidate.shape) == FULL_SIZE:
            return np.ascontiguousarray(
                candidate,
                dtype=np.float32,
            )

    raise ValueError(
        f"{source_name}: {array.shape} を "
        f"{FULL_SIZE} に変換できません。"
    )


def load_npz_image(npz_path):
    with np.load(
        npz_path,
        allow_pickle=False,
    ) as archive:

        key = (
            "Train"
            if "Train" in archive.files
            else archive.files[0]
        )

        return to_dhw(
            archive[key],
            f"{npz_path.name} ({key})",
        )


# ------------------------------------------------------------
# 4. Metrics
# ------------------------------------------------------------

def select_slices(volume):
    start = (
        0
        if EVAL_SLICE_START is None
        else max(0, EVAL_SLICE_START)
    )

    end = (
        volume.shape[0]
        if EVAL_SLICE_END is None
        else min(
            volume.shape[0],
            EVAL_SLICE_END + 1,
        )
    )

    if start >= end:
        raise ValueError(
            "EVAL_SLICE_START / EVAL_SLICE_ENDが不正です。"
        )

    return volume[start:end], start, end - 1


def compute_ncc(fixed, moved):
    fixed = fixed.ravel().astype(np.float64)
    moved = moved.ravel().astype(np.float64)

    fixed = fixed - fixed.mean()
    moved = moved - moved.mean()

    return float(
        np.dot(fixed, moved)
        / (
            np.linalg.norm(fixed)
            * np.linalg.norm(moved)
            + 1e-12
        )
    )


def compute_dice_jaccard(fixed, moved):
    fixed_mask = (
        fixed > FOREGROUND_THRESHOLD
    )

    moved_mask = (
        moved > FOREGROUND_THRESHOLD
    )

    intersection = np.logical_and(
        fixed_mask,
        moved_mask,
    ).sum(dtype=np.float64)

    union = np.logical_or(
        fixed_mask,
        moved_mask,
    ).sum(dtype=np.float64)

    dice = (
        2.0 * intersection
        / (
            fixed_mask.sum()
            + moved_mask.sum()
            + 1e-12
        )
    )

    jaccard = (
        intersection
        / (union + 1e-12)
    )

    return float(dice), float(jaccard)


def compute_ms_ssim_fsim(fixed, moved):
    """
    MS-SSIM: 全axial sliceをbatchとして評価
    FSIM: 中央sliceで評価
    """
    fixed_tensor = torch.from_numpy(
        fixed
    ).unsqueeze(1).to(
        device=device,
        dtype=torch.float32,
    )

    moved_tensor = torch.from_numpy(
        moved
    ).unsqueeze(1).to(
        device=device,
        dtype=torch.float32,
    )

    # piqは厳密に[0, 1]を要求する
    fixed_tensor = fixed_tensor.clamp(
        0.0,
        1.0,
    )

    moved_tensor = moved_tensor.clamp(
        0.0,
        1.0,
    )

    ms_ssim_value = multi_scale_ssim(
        moved_tensor,
        fixed_tensor,
        data_range=1.0,
    ).item()

    middle_slice = fixed.shape[0] // 2

    fsim_value = fsim(
        moved_tensor[
            middle_slice:middle_slice + 1
        ],
        fixed_tensor[
            middle_slice:middle_slice + 1
        ],
        data_range=1.0,
        chromatic=False,
    ).item()

    return float(ms_ssim_value), float(fsim_value)


def compute_jacobian_metrics(flow):
    """
    flow: (B, 3, D, H, W)
    folding_percentが0%に近いほどよい。
    """
    uz = flow[:, 0].float()
    uy = flow[:, 1].float()
    ux = flow[:, 2].float()

    duz_dz, duz_dy, duz_dx = torch.gradient(
        uz,
        dim=(1, 2, 3),
    )

    duy_dz, duy_dy, duy_dx = torch.gradient(
        uy,
        dim=(1, 2, 3),
    )

    dux_dz, dux_dy, dux_dx = torch.gradient(
        ux,
        dim=(1, 2, 3),
    )

    determinant = (
        (1.0 + duz_dz)
        * (
            (1.0 + duy_dy)
            * (1.0 + dux_dx)
            - duy_dx * dux_dy
        )
        - duz_dy
        * (
            duy_dz * (1.0 + dux_dx)
            - duy_dx * dux_dz
        )
        + duz_dx
        * (
            duy_dz * dux_dy
            - (1.0 + duy_dy) * dux_dz
        )
    )

    return {
        "folding_percent": float(
            100.0
            * (
                determinant <= 0
            ).float().mean().item()
        ),
        "jacobian_mean": float(
            determinant.mean().item()
        ),
        "jacobian_min": float(
            determinant.min().item()
        ),
    }


def compute_inverse_consistency_mse(
    flow_m2f_low,
    flow_f2m_low,
):
    composed_m2f = (
        flow_m2f_low
        + transformer_low(
            flow_f2m_low,
            flow_m2f_low,
        )
    )

    composed_f2m = (
        flow_f2m_low
        + transformer_low(
            flow_m2f_low,
            flow_f2m_low,
        )
    )

    return float(
        0.5
        * (
            composed_m2f.square().mean()
            + composed_f2m.square().mean()
        ).item()
    )


# ------------------------------------------------------------
# 5. Evaluate one pair
# ------------------------------------------------------------

@torch.inference_mode()
def evaluate_pair(model, pair_dir, model_name):
    npz_files = sorted(
        pair_dir.glob("*.npz")
    )

    if len(npz_files) != 2:
        raise ValueError(
            f"{pair_dir.name}: .npzファイルは"
            f"2個必要です。実際={len(npz_files)}個"
        )

    fixed_path = npz_files[
        FIXED_FILE_INDEX
    ]

    moving_path = npz_files[
        MOVING_FILE_INDEX
    ]

    fixed_np = load_npz_image(
        fixed_path
    )

    moving_np = load_npz_image(
        moving_path
    )

    fixed = torch.from_numpy(
        fixed_np
    ).unsqueeze(0).unsqueeze(0).to(
        device=device,
        dtype=torch.float32,
    )

    moving = torch.from_numpy(
        moving_np
    ).unsqueeze(0).unsqueeze(0).to(
        device=device,
        dtype=torch.float32,
    )

    moving_w = wavelet_input(
        moving
    )

    fixed_w = wavelet_input(
        fixed
    )

    flow_m2f_low = model(
        moving_w,
        fixed_w,
    )

    flow_f2m_low = model(
        fixed_w,
        moving_w,
    )

    flow_m2f_full = (
        low_dvf_to_full_trilinear(
            flow_m2f_low
        )
    )

    moved = transformer_full(
        moving,
        flow_m2f_full,
    )

    moved_np = (
        moved[0, 0]
        .float()
        .cpu()
        .numpy()
    )

    fixed_eval, slice_start, slice_end = (
        select_slices(fixed_np)
    )

    moved_eval, _, _ = select_slices(
        moved_np
    )

    error = moved_eval - fixed_eval

    mse = float(
        np.mean(error ** 2)
    )

    rmse_normalized = float(
        np.sqrt(mse)
    )

    dice, jaccard = compute_dice_jaccard(
        fixed_eval,
        moved_eval,
    )

    ms_ssim_value, fsim_value = (
        compute_ms_ssim_fsim(
            fixed_eval,
            moved_eval,
        )
    )

    jacobian = compute_jacobian_metrics(
        flow_m2f_full
    )

    return {
        "model": model_name,
        "pair_id": pair_dir.name,
        "fixed_file": fixed_path.name,
        "moving_file": moving_path.name,
        "slice_start": slice_start,
        "slice_end": slice_end,
        "dice": dice,
        "jaccard": jaccard,
        "ms_ssim": ms_ssim_value,
        "fsim": fsim_value,
        "mse": mse,
        "rmse_hu": (
            rmse_normalized
            * (CT_MAX - CT_MIN)
        ),
        "ncc": compute_ncc(
            fixed_eval,
            moved_eval,
        ),
        "folding_percent": jacobian[
            "folding_percent"
        ],
        "jacobian_mean": jacobian[
            "jacobian_mean"
        ],
        "jacobian_min": jacobian[
            "jacobian_min"
        ],
        "inverse_consistency_mse_low": (
            compute_inverse_consistency_mse(
                flow_m2f_low,
                flow_f2m_low,
            )
        ),
    }


# ------------------------------------------------------------
# 6. Evaluate all candidates
# ------------------------------------------------------------

pair_dirs = sorted(
    path
    for path in TEST_DATA_ROOT.iterdir()
    if path.is_dir()
)

if not pair_dirs:
    raise RuntimeError(
        f"テストペアがありません: "
        f"{TEST_DATA_ROOT}"
    )

all_candidate_summaries = []

for model_name, model_path in (
    CANDIDATE_MODELS.items()
):

    print("\n" + "=" * 80)
    print(
        f"TEST EVALUATION: {model_name}"
    )
    print("=" * 80)

    try:
        checkpoint = torch.load(
            model_path,
            map_location=device,
            weights_only=True,
        )
    except TypeError:
        checkpoint = torch.load(
            model_path,
            map_location=device,
        )

    state_dict = checkpoint.get(
        "model_state_dict",
        checkpoint,
    )

    model = create_model()

    model.load_state_dict(
        state_dict,
        strict=True,
    )

    model.eval()

    pair_results = []

    for index, pair_dir in enumerate(
        pair_dirs,
        start=1,
    ):
        result = evaluate_pair(
            model,
            pair_dir,
            model_name,
        )

        pair_results.append(result)

        print(
            f"[{index:03d}/{len(pair_dirs):03d}] "
            f"{result['pair_id']} | "
            f"Dice={result['dice']:.5f} | "
            f"NCC={result['ncc']:.5f} | "
            f"Folding="
            f"{result['folding_percent']:.4f}%"
        )

    # --------------------------------------------------------
    # Pair-wise table
    # --------------------------------------------------------

    print(
        f"\n成功した結果数: {len(pair_results)}"
    )

    print(
        f"\n=== Pair-wise results: "
        f"{model_name} ==="
    )

    table_rows = [
        [
            row["pair_id"],
            row["dice"],
            row["jaccard"],
            row["ms_ssim"],
            row["fsim"],
            row["rmse_hu"],
            row["ncc"],
            row["folding_percent"],
            row[
                "inverse_consistency_mse_low"
            ],
        ]
        for row in pair_results
    ]

    print(
        tabulate(
            table_rows,
            headers=[
                "pair_id",
                "Dice",
                "Jaccard",
                "MS-SSIM",
                "FSIM",
                "RMSE [HU]",
                "NCC",
                "Folding [%]",
                "IC MSE",
            ],
            floatfmt=".6f",
            tablefmt="github",
        )
    )

    # --------------------------------------------------------
    # Mean / SD / max Folding
    # --------------------------------------------------------

    metric_keys = [
        "dice",
        "jaccard",
        "ms_ssim",
        "fsim",
        "rmse_hu",
        "ncc",
        "folding_percent",
        "inverse_consistency_mse_low",
    ]

    values = {
        key: np.asarray(
            [
                row[key]
                for row in pair_results
            ],
            dtype=np.float64,
        )
        for key in metric_keys
    }

    folding_values = values[
        "folding_percent"
    ]

    worst_index = int(
        np.argmax(folding_values)
    )

    worst_row = pair_results[
        worst_index
    ]

    print(
        f"\n=== {model_name}: "
        f"mean ± SD ==="
    )

    for key in metric_keys:
        print(
            f"{key:30s}: "
            f"{values[key].mean():.6f} ± "
            f"{values[key].std(ddof=1):.6f}"
        )

    print(
        "\n=== Folding summary ==="
    )

    print(
        f"mean folding: "
        f"{folding_values.mean():.6f}%"
    )

    print(
        f"max folding : "
        f"{folding_values.max():.6f}%"
    )

    print(
        f"worst pair  : "
        f"{worst_row['pair_id']}"
    )

    # --------------------------------------------------------
    # Save per-candidate CSV
    # --------------------------------------------------------

    candidate_csv_path = (
        RESULT_ROOT
        / f"{model_name}_test_metrics.csv"
    )

    with candidate_csv_path.open(
        "w",
        newline="",
        encoding="utf-8-sig",
    ) as file:

        writer = csv.DictWriter(
            file,
            fieldnames=list(
                pair_results[0].keys()
            ),
        )

        writer.writeheader()
        writer.writerows(pair_results)

    all_candidate_summaries.append(
        {
            "model": model_name,
            "model_path": str(
                model_path.resolve()
            ),
            "dice_mean": values[
                "dice"
            ].mean(),
            "dice_sd": values[
                "dice"
            ].std(ddof=1),
            "jaccard_mean": values[
                "jaccard"
            ].mean(),
            "ms_ssim_mean": values[
                "ms_ssim"
            ].mean(),
            "fsim_mean": values[
                "fsim"
            ].mean(),
            "rmse_hu_mean": values[
                "rmse_hu"
            ].mean(),
            "ncc_mean": values[
                "ncc"
            ].mean(),
            "folding_mean_percent": (
                folding_values.mean()
            ),
            "folding_max_percent": (
                folding_values.max()
            ),
            "worst_folding_pair": worst_row[
                "pair_id"
            ],
            "ic_mse_mean": values[
                "inverse_consistency_mse_low"
            ].mean(),
            "per_pair_csv": str(
                candidate_csv_path.resolve()
            ),
        }
    )

    del model
    del checkpoint

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ------------------------------------------------------------
# 7. Final comparison table / CSV
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL SMOOTHNESS CANDIDATE COMPARISON")
print("=" * 80)

comparison_rows = [
    [
        row["model"],
        row["dice_mean"],
        row["jaccard_mean"],
        row["ms_ssim_mean"],
        row["fsim_mean"],
        row["rmse_hu_mean"],
        row["ncc_mean"],
        row["folding_mean_percent"],
        row["folding_max_percent"],
        row["worst_folding_pair"],
        row["ic_mse_mean"],
    ]
    for row in all_candidate_summaries
]

print(
    tabulate(
        comparison_rows,
        headers=[
            "Model",
            "Dice",
            "Jaccard",
            "MS-SSIM",
            "FSIM",
            "RMSE [HU]",
            "NCC",
            "Mean Folding [%]",
            "Max Folding [%]",
            "Worst pair",
            "IC MSE",
        ],
        floatfmt=".6f",
        tablefmt="github",
    )
)

comparison_csv_path = (
    RESULT_ROOT
    / "smoothness_candidate_comparison.csv"
)

with comparison_csv_path.open(
    "w",
    newline="",
    encoding="utf-8-sig",
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=list(
            all_candidate_summaries[0].keys()
        ),
    )

    writer.writeheader()
    writer.writerows(
        all_candidate_summaries
    )

print("\n保存先:")
print(comparison_csv_path.resolve())

Device: cuda:0
TestData: D:\Saito\Data\TestData
Result directory: D:\Saito\fullres_trilinear_ic_beta1e3_80k\finetune_smoothness_comparison\testdata_comparison

TEST EVALUATION: smooth_1e-03
[64, 128, 128]
[001/101] pair1 | Dice=0.95401 | NCC=0.99343 | Folding=0.1741%
[002/101] pair10 | Dice=0.95382 | NCC=0.99327 | Folding=0.1893%
[003/101] pair100 | Dice=0.94978 | NCC=0.99386 | Folding=0.0674%
[004/101] pair101 | Dice=0.96678 | NCC=0.99443 | Folding=0.1432%
[005/101] pair11 | Dice=0.95121 | NCC=0.99266 | Folding=0.1004%
[006/101] pair12 | Dice=0.95838 | NCC=0.97553 | Folding=1.2743%
[007/101] pair13 | Dice=0.94059 | NCC=0.99530 | Folding=0.0514%
[008/101] pair14 | Dice=0.96018 | NCC=0.99520 | Folding=0.0500%
[009/101] pair15 | Dice=0.96346 | NCC=0.99286 | Folding=0.0810%
[010/101] pair16 | Dice=0.96924 | NCC=0.99243 | Folding=0.0858%
[011/101] pair17 | Dice=0.96975 | NCC=0.99167 | Folding=0.2829%
[012/101] pair18 | Dice=0.96765 | NCC=0.97795 | Folding=1.5851%
[013/101] pair19 | Dice=0.